In [ ]:
import torch
from src.model import CODI
from src.datasets import extract_answer_number

# Minimal helper copied from experiments: add pad and CoDI special tokens

def ensure_tokenizer_special_tokens(tokenizer, model) -> None:
    if tokenizer.pad_token_id is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        tokenizer.pad_token_id = model.pad_token_id
    tokenizer.add_special_tokens({"additional_special_tokens": ["<|bocot|>", "<|eocot|>"]})

checkpoint = "bcywinski/codi_llama1b-answer_only"
base_model = "meta-llama/Llama-3.2-1B-Instruct"  # or whatever you used
device = "cuda"
dtype = "bfloat16"

model = CODI.from_pretrained(
    checkpoint_path=checkpoint,
    model_name_or_path=base_model,
    lora_r=128,
    lora_alpha=32,
    num_latent=6,
    use_prj=True,
    device=device,
    dtype=dtype,
    strict=False,
    checkpoint_save_path=f"./checkpoints/{checkpoint}",
    remove_eos=False,
    full_precision=True,
)
tokenizer = model.tokenizer
ensure_tokenizer_special_tokens(tokenizer, model)

prompt = "A team starts with 3 members. They recruit 1 new member. Then each current member recruits 2 additional people. How many people are there now on the team? Give the answer only and nothing else."

num_latent_iterations = 6  # try 0 to skip thinking, or 1..6
output = model.generate(
    **tokenizer(prompt, return_tensors="pt").to(model.codi.device),
    tokenizer=tokenizer,
    max_new_tokens=64,
    num_latent_iterations=num_latent_iterations,
    temperature=1.0,
    greedy=False,
    return_latent_vectors=True,
    remove_eos=False,
    output_hidden_states=True,
    output_attentions=False,
    skip_thinking=(num_latent_iterations == 0),
    sot_token=tokenizer.convert_tokens_to_ids("<|bocot|>"),
    eot_token=tokenizer.convert_tokens_to_ids("<|eocot|>"),
    verbalize_cot=False,  # must be False to get latent_vectors (True short-circuits)
)

text = tokenizer.decode(output["sequences"][0], skip_special_tokens=False)
print("Generated text:\n", text)
print("Parsed answer:", extract_answer_number(text))

latents = output.get("latent_vectors")  # list/array per iteration if returned
if latents is not None:
    print("Latent iterations:", len(latents))
    print("Shape of first latent block:", latents[0].shape)
else:
    print("No latent vectors returned (check verbalize_cot/skip_thinking settings).")


Checkpoint: bcywinski/codi_llama1b-answer_only (HuggingFace ID)
  → Checkpoint already exists at: ./checkpoints/bcywinski/codi_llama1b-answer_only
  → Skipping download
Loading checkpoint from ./checkpoints/bcywinski/codi_llama1b-answer_only/pytorch_model.bin
Model loaded successfully from ./checkpoints/bcywinski/codi_llama1b-answer_only
Generated text:
 12<|eot_id|>
Parsed answer: 12.0
Latent iterations: 7
Shape of first latent block: torch.Size([1, 1, 2048])


: 